# Ring-Light Best Stack — Cheek Lab Inference (Colab)

Production-style **chart-free** cheek Lab from a Variable Lighting **torch zip** (no-flash + flash DNGs + Apple landmarks).

## Frozen color stack (n=84 ring-light eval)

| Step | What |
|---|---|
| 1 | Pre-AWB demosaic → reflectance \(R_0=\sqrt{A_0\odot B_0'}\) |
| 2 | Apple Vision cheek mask |
| 3 | **`tier3_affine`** indoor RGB→XYZ (not ring CC affine) |
| 4 | **`hybrid_deploy` CAT** — Lu+torch SPD on F12/warm; frozen 5500 K on D65 |
| 5 | **Illuminant-routed multi-Lab corrector** (`W_d65` / `W_f12`) |
| 6 | FairFace-7 → specular/shadow cheek sampling → Lab |

Pinned ring-light accuracy vs FitSkin forehead Lab: **mean ΔE₀₀ ≈ 8.5** (vs ~9.8 frozen 5500 K alone).

### How to run
1. **Runtime → GPU** optional (FairFace runs fine on CPU)
2. Run cells top → bottom
3. **Cell 2** — clone repo + verify calibration bundles
4. **Cell 2b** — downloads 4 demo zips from GitHub (Anjana + Lihn)
5. **Cell 4** — one-zip inference + cheek / Lab swatch
6. **Cell 5** — optional batch over a folder

Local CLI equivalent:
```bash
python scripts/run_d65_fairface7_roi.py \
  --zip path/to/AnjanaF12B1Torch.zip \
  --cat-mode hybrid_deploy \
  --multi-lab-corrector calibration/multi_illuminant_lab_affine/multi_illuminant_lab_affine.json
```

> Torch SPD: uses `Torch_meas/` if present; otherwise falls back to flash SPD baked into `tier3_affine/iphone_calibration_bundle.json` (~4923 K).

## 0 — Setup


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q rawpy opencv-python-headless numpy matplotlib gdown

try:
    import torch, torchvision  # noqa: F401
except ImportError:
    !pip install -q torch torchvision

import json, sys, zipfile
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = "https://github.com/RooneyEmily/Fitskin.git"
if Path("Fitskin").is_dir():
    !cd Fitskin && git pull --ff-only 2>/dev/null || true
    REPO = Path("Fitskin").resolve()
elif (Path.cwd() / "pipeline" / "d65_fairface7_roi.py").is_file():
    REPO = Path.cwd().resolve()
else:
    !git clone -q {REPO_URL}
    REPO = Path("Fitskin").resolve()

if IN_COLAB and not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

sys.path = [str(REPO)] + [p for p in sys.path if Path(p).resolve() != REPO]

# Ensure best-stack calibration is present (may not be on remote git tip yet)
MULTI_LAB = REPO / "calibration" / "multi_illuminant_lab_affine" / "multi_illuminant_lab_affine.json"
PIPE = REPO / "pipeline" / "d65_fairface7_roi.py"
ASSET_ZIP = REPO / "colab_assets" / "ringlight_best_stack.zip"
if (not MULTI_LAB.is_file()) or (not PIPE.is_file()):
    if ASSET_ZIP.is_file():
        print("Extracting colab_assets/ringlight_best_stack.zip …")
        with zipfile.ZipFile(ASSET_ZIP) as zf:
            zf.extractall(REPO)
    else:
        print("WARN: missing multi-lab bundle — push colab_assets/ringlight_best_stack.zip to repo")

CAL_DIR = REPO / "calibration" / "tier3_affine"
FAIRFACE_DIR = REPO / "calibration" / "fairface"
FAIRFACE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("/content/ringlight_best_stack_runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FF7 = FAIRFACE_DIR / "res34_fair_align_multi_7_20190809.pt"
if not FF7.is_file():
    print("Downloading FairFace-7 weights (~82 MB)…")
    !gdown 11y0Wi3YQf21a_VcspUV4FwqzhMcfaVAB -O "{FF7}"
assert FF7.is_file(), "FairFace weights missing"

import torch
from pipeline.d65_fairface7_roi import D65FairFace7ROIPipeline, write_result_json

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("REPO:", REPO)
print("tier3 affine:", (CAL_DIR / "camera_rgb_to_xyz_affine.npy").is_file())
print("multi-lab corrector:", MULTI_LAB.is_file())
print("Setup OK.")

## 1 — Input zip

Pick **one** of:
- **Upload** a `*Torch.zip` (Cell 2a)
- **Drive path** to ring-light folder (Cell 2b)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2a — OPTIONAL: upload one torch zip
# ══════════════════════════════════════════════════════════════════════════════
UPLOAD_DIR = Path("/content/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

DO_UPLOAD = False  # True = pick one *Torch.zip in the widget below
print("Set DO_UPLOAD=True then click Choose Files below.")
if DO_UPLOAD and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        (UPLOAD_DIR / name).write_bytes(data)
        print("saved", UPLOAD_DIR / name)
else:
    print("Upload skipped — use Cell 2b Drive path or set DO_UPLOAD=True")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2b — Demo zips from GitHub repo (no Drive / upload needed)
# ══════════════════════════════════════════════════════════════════════════════
import json
import urllib.request

from pipeline.illuminant_estimation import infer_illuminant_label

DEMO_PERSON = "Anjana"       # "Anjana" | "Lihn"
PREFER_ILLUMINANT = "F12"    # "F12" or "D65"

DEMO_DIR = REPO / "data" / "ring_light" / "demo_zips"
MANIFEST_PATH = REPO / "data" / "ring_light" / "demo_manifest.json"
GITHUB_RAW = "https://github.com/RooneyEmily/Fitskin/raw/main/data/ring_light/demo_zips"

TORCH_DIR = Path("/content/drive/MyDrive/Torch_meas")  # optional

DEMO_DIR.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.is_file():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
else:
    manifest = {
        "demos": [
            {"file": "AnjanaF12B1Torch.zip", "person": "Anjana", "illuminant": "F12"},
            {"file": "Anjana-D65-C3Torch.zip", "person": "Anjana", "illuminant": "D65"},
            {"file": "Lihn-F12-B1Torch.zip", "person": "Lihn", "illuminant": "F12"},
            {"file": "LihnD65-C1Torch.zip", "person": "Lihn", "illuminant": "D65"},
        ]
    }

for d in manifest.get("demos", []):
    name = d["file"]
    dest = DEMO_DIR / name
    if dest.is_file() and dest.stat().st_size > 1_000_000:
        print("have", name)
        continue
    url = f"{GITHUB_RAW}/{name}"
    print(f"Downloading {name} (~20 MB) …")
    urllib.request.urlretrieve(url, dest)
    print("  ->", dest, f"({dest.stat().st_size/1e6:.1f} MB)")

candidates = sorted(DEMO_DIR.glob("*.zip"))
# optional: also pick up manual uploads
if UPLOAD_DIR.is_dir():
    candidates = sorted(set(candidates) | set(UPLOAD_DIR.glob("*.zip")), key=lambda p: p.name)

print(f"\nDemo zips ready ({len(candidates)}):")
for p in candidates:
    print(f"  [{infer_illuminant_label(p) or '?'}] {p.name}")

def _person_match(path: Path, person: str) -> bool:
    s = path.name.lower()
    aliases = {"anjana": ("anjana",), "lihn": ("lihn", "linh")}
    keys = aliases.get(person.lower(), (person.lower(),))
    return any(k in s for k in keys)

person_pool = [p for p in candidates if _person_match(p, DEMO_PERSON)]
if not person_pool:
    raise RuntimeError(f"No demo zip for {DEMO_PERSON!r} in {DEMO_DIR}")

ill = PREFER_ILLUMINANT.upper()
ZIP_PATH = next((p for p in person_pool if infer_illuminant_label(p) == ill), person_pool[0])

print(f"\nSelected ({DEMO_PERSON}, {PREFER_ILLUMINANT}):", ZIP_PATH)
print("Inferred illuminant:", infer_illuminant_label(ZIP_PATH))


## 2 — Load best-stack pipeline + run one zip


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Best stack inference + visuals
# ══════════════════════════════════════════════════════════════════════════════
import cv2
import matplotlib.pyplot as plt
import numpy as np
from scripts.evaluate_pansor20_chartfree_d65 import (
    apple_face_cheek_masks,
    extract_zip,
    linear_rgb_to_preview_bgr,
    load_apple_landmarks,
    load_dng_linear,
)
from models.fairface_race import face_rgb_crop_from_landmarks

CAT_MODE = "hybrid_deploy"
SAMPLING = "fairface7"  # "off" = trimmed mean only

pipe = D65FairFace7ROIPipeline.from_defaults(
    cal_dir=CAL_DIR,
    fairface_dir=FAIRFACE_DIR,
    cat_mode=CAT_MODE,
    torch_dir=TORCH_DIR if TORCH_DIR.is_dir() else None,
    multi_lab_affine=MULTI_LAB,
    half_size=True,
    sampling=SAMPLING,
)
print(f"Pipeline: cat_mode={CAT_MODE}  multi_lab={MULTI_LAB.name}  sampling={SAMPLING}")

result = pipe.run_zip(ZIP_PATH)
out_json = OUT_DIR / f"{ZIP_PATH.stem}.json"
write_result_json(result, out_json)

_cat = result.get("cat_cct")
_cat_s = f"{float(_cat):.0f} K" if _cat is not None else "?"
_lu = result.get("lu_cct_k")
_lu_s = f"{float(_lu):.0f} K" if _lu is not None else "n/a"
print(
    f"\nLab = ({result['L']:.2f}, {result['a']:.2f}, {result['b']:.2f})\n"
    f"illuminant = {result.get('illuminant_label')}  "
    f"CAT CCT = {_cat_s}  Lu CCT = {_lu_s}\n"
    f"lab_corrector = {result.get('lab_corrector')}  "
    f"FairFace = {result.get('fairface_label')} → {result.get('predicted_ethnicity')} "
    f"(conf={result.get('fairface_confidence'):.2f})\n"
    f"n_cheek = {result.get('n_cheek')}  flash_scale = {result.get('flash_scale'):.3f}"
)
ef = result.get("exposure_flags") or {}
if ef.get("out_of_band"):
    print("⚠ exposure_flags:", ef)
else:
    print("exposure_flags: OK")
print("Wrote", out_json)

# ── visuals ─────────────────────────────────────────────────────────────────
work = OUT_DIR / "_viz"
nf, fl, lm_path = extract_zip(ZIP_PATH, work / ZIP_PATH.stem)
A0 = load_dng_linear(nf, half_size=True, use_camera_wb=False)
lm = load_apple_landmarks(lm_path)
_, cheek = apple_face_cheek_masks(lm, A0.shape[0], A0.shape[1])
preview = linear_rgb_to_preview_bgr(A0)
overlay = preview.copy()
overlay[cheek > 0] = (0.55 * overlay[cheek > 0] + 0.45 * np.array([0, 220, 80])).astype(np.uint8)
face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)

def lab_to_srgb_u8(L, a, b):
    fy = (L + 16.0) / 116.0
    fx, fz = fy + a / 500.0, fy - b / 200.0
    eps, kappa = 216 / 24389, 24389 / 27
    def finv(t):
        return t**3 if t**3 > eps else (116 * t - 16) / kappa
    X, Y, Z = 0.95047 * finv(fx), finv(fy), 1.08883 * finv(fz)
    M = np.array([[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])
    rgb = M @ np.array([X, Y, Z])
    lin2s = lambda u: 12.92 * u if u <= 0.0031308 else 1.055 * (max(u, 0) ** (1 / 2.4)) - 0.055
    return (np.clip([lin2s(float(c)) for c in rgb], 0, 1) * 255).astype(np.uint8)

swatch = np.full((180, 180, 3), lab_to_srgb_u8(result["L"], result["a"], result["b"]), dtype=np.uint8)

fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
ax[0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)); ax[0].set_title("No-flash"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); ax[1].set_title(f"Cheek ROI (n={result['n_cheek']})"); ax[1].axis("off")
ax[2].imshow(face_rgb); ax[2].set_title(f"FairFace-7\n{result.get('fairface_label')} → {result.get('predicted_ethnicity')}"); ax[2].axis("off")
ax[3].imshow(swatch); ax[3].set_title(f"Cheek Lab\n({result['L']:.1f}, {result['a']:.1f}, {result['b']:.1f})"); ax[3].axis("off")
plt.suptitle(f"{ZIP_PATH.name}  ·  hybrid_deploy + multi-lab  ·  {result.get('illuminant_label')}", fontsize=11)
plt.tight_layout()
plt.show()

## 3 — Optional: compare frozen vs best stack on the same zip


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Side-by-side: frozen 5500 K vs best stack
# ══════════════════════════════════════════════════════════════════════════════
pipe_frozen = D65FairFace7ROIPipeline.from_defaults(
    cal_dir=CAL_DIR,
    fairface_dir=FAIRFACE_DIR,
    cat_mode="frozen_5500",
    half_size=True,
    sampling=SAMPLING,
)
r_frozen = pipe_frozen.run_zip(ZIP_PATH)

print(f"{'Arm':<22} {'L*':>7} {'a*':>7} {'b*':>7} {'CAT K':>8} {'corrector':<12}")
print("-" * 70)
for label, r in [("frozen_5500 (baseline)", r_frozen), ("best stack", result)]:
    print(
        f"{label:<22} {r['L']:7.2f} {r['a']:7.2f} {r['b']:7.2f} "
        f"{float(r.get('cat_cct') or 0):8.0f} {str(r.get('lab_corrector') or '-'):<12}"
    )
dL, da, db = result["L"] - r_frozen["L"], result["a"] - r_frozen["a"], result["b"] - r_frozen["b"]
print(f"\nΔLab (best − frozen): ΔL*={dL:+.2f}  Δa*={da:+.2f}  Δb*={db:+.2f}")

## 4 — Optional batch


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — OPTIONAL batch (reuse `pipe` from Cell 3)
# ══════════════════════════════════════════════════════════════════════════════
RUN_BATCH = False  # set True to process all zips found in Cell 2b

if RUN_BATCH:
    summary = []
    for i, zp in enumerate(candidates, 1):
        try:
            r = pipe.run_zip(zp)
        except Exception as exc:
            print(f"[{i:02d}/{len(candidates)}] FAIL {zp.name}: {exc}")
            summary.append({"zip": str(zp), "error": str(exc)})
            continue
        out = OUT_DIR / f"{zp.stem}.json"
        write_result_json(r, out)
        print(
            f"[{i:02d}/{len(candidates)}] {zp.name:36s}  "
            f"Lab=({r['L']:.1f},{r['a']:.1f},{r['b']:.1f})  "
            f"ill={r.get('illuminant_label')}  cat={r.get('cat_cct'):.0f}K  "
            f"FF={r.get('fairface_label')}→{r.get('predicted_ethnicity')}"
        )
        summary.append({
            "zip": str(zp), "out": str(out),
            "L": r["L"], "a": r["a"], "b": r["b"],
            "illuminant": r.get("illuminant_label"),
            "cat_cct": r.get("cat_cct"),
            "lab_corrector": r.get("lab_corrector"),
            "fairface_label": r.get("fairface_label"),
            "predicted_ethnicity": r.get("predicted_ethnicity"),
        })
    batch_summary = OUT_DIR / "batch_summary.json"
    batch_summary.write_text(json.dumps({"n": len(summary), "trials": summary}, indent=2) + "\n")
    print("Wrote", batch_summary)
else:
    print("Batch skipped (set RUN_BATCH=True).")

## Reference — pinned ring-light eval (n=84)

| Arm | All mean ΔE₀₀ | D65 ring | F12 ring |
|---|---:|---:|---:|
| frozen_5500 | 9.75 | 6.46 | 13.37 |
| hybrid_deploy | 9.28 | 6.81 | 12.00 |
| **hybrid_deploy + multi-lab** | **~8.5** | **~8.0** | **~9.1** |

**Do not use** ring CC-supervised affines for cheek — chart patches fit (~8 ΔE) but cheek stays ~30 ΔE.

Rebuild Colab asset zip locally:
```bash
python scripts/build_ringlight_colab_assets.py
```
